# Static Preprocessing  
Conversione da csv --> parquet  
Unione in unico file  
Eliminazione duplicati  
Eliminazione valori nulli  
Togliere tutti i valori di MMSI che non rappresentano delle vere navi  
Togliere i valori di default del protocollo AIS che potrebbero influenzare negativamente la rete neurale  
Togliere tutte le navi ferme  

In [1]:
import pandas as pd
import glob
import os
import polars as pl
import numpy as np

input_path = '../../data/raw/' # Cartella contenente i file CSV originali
final_output = '../../data/processed/' # Cartella di destinazione per i file Parquet finali

# Crea le cartelle se non esistono
os.makedirs('../../data/processed/', exist_ok=True)

keep_cols = ['mmsi','base_date_time','longitude','latitude','sog','cog','heading','vessel_type']

# Lista dei file CSV
files = sorted(glob.glob(os.path.join(input_path, "ais-*.csv")))
print(f"File trovati: {len(files)}")

File trovati: 63


In [2]:
# Raggruppiamo i file in blocchi da 1 settimana
giorni_per_settimana = 7
settimane = [files[i:i + giorni_per_settimana] for i in range(0, len(files), giorni_per_settimana)]

for indice_settimana, file_settimana in enumerate(settimane):
    nome_output = os.path.join(final_output, f"ais_settimana_{indice_settimana + 1}.parquet")
    
    if os.path.exists(nome_output):
        print(f"Settimana {indice_settimana + 1} già processata. Salto")
        continue
        
    print(f"Processando Settimana {indice_settimana + 1} ({len(file_settimana)} file CSV)")
    
    # Polars prepara le istruzioni senza saturare la RAM
    lazy_df = pl.scan_csv(file_settimana)
    
    # Pipelines di pulizia e feature engineering
    processed_lazy = (
        lazy_df
        # Rimuovi duplicati (se presenti) e valori nulli critici
        .unique()
        .drop_nulls(subset=['mmsi', 'latitude', 'longitude', 'sog', 'cog', 'heading'])
        
        # Filtri fisici e di protocollo AIS
        .filter(
            (pl.col("latitude").is_between(-90.0, 90.0)) &
            (pl.col("longitude").is_between(-180.0, 180.0)) &
            (pl.col("mmsi").is_between(201000000, 775999999)) & # Filtro MMSI navi standard
            (pl.col("sog") < 102.3) &                           # Esclusione placeholder SOG
            (pl.col("cog") < 360.0) &                           # Esclusione placeholder COG
            (pl.col("heading") < 360.0) &                       # Esclusione placeholder HEADING
            (pl.col("sog") >= 1.0)                              # Esclusione navi ferme
        )
        
        # Trasformazione Trigonometrica per COG e HEADING
        .with_columns(
            cog_rad = pl.col("cog") * (np.pi / 180.0),
            heading_rad = pl.col("heading") * (np.pi / 180.0)
        )
        .with_columns(
            cog_cos = pl.col("cog_rad").cos(),
            cog_sin = pl.col("cog_rad").sin(),
            heading_cos = pl.col("heading_rad").cos(),
            heading_sin = pl.col("heading_rad").sin()
        )
        # Rimozione colonne temporanee e non più necessarie in gradi
        .drop(["cog", "heading", "cog_rad", "heading_rad"])
        
        # Filtro traiettorie brevi: mantieni solo MMSI con almeno 30 messaggi
        .filter(
            pl.col("mmsi").count().over("mmsi") >= 30
        )
        .select([
            "mmsi", 
            "base_date_time", 
            "latitude", 
            "longitude", 
            "sog", 
            "cog_cos", 
            "cog_sin", 
            "heading_cos", 
            "heading_sin" 
        ])
        
    )
    
    #Usiamo collect solo alla fine per salvare il DataFrame pulito
    df_settimana = processed_lazy.collect()
    df_settimana = df_settimana.sort(["mmsi", "base_date_time"])
    
    df_settimana.write_parquet(nome_output)
    print(f"Salvato")

print("Preprocessing statico completato")

Processando Settimana 1 (7 file CSV)
Salvato
Processando Settimana 2 (7 file CSV)
Salvato
Processando Settimana 3 (7 file CSV)
Salvato
Processando Settimana 4 (7 file CSV)
Salvato
Processando Settimana 5 (7 file CSV)
Salvato
Processando Settimana 6 (7 file CSV)
Salvato
Processando Settimana 7 (7 file CSV)
Salvato
Processando Settimana 8 (7 file CSV)
Salvato
Processando Settimana 9 (7 file CSV)
Salvato
Preprocessing statico completato


In [4]:
# Puntiamo a TUTTI i file parquet presenti nella cartella
percorso_cartella = '../../data/processed/*.parquet' 
lazy_df = pl.scan_parquet(percorso_cartella)

# Calcoliamo le statistiche globali in streaming
stats = lazy_df.select([
    pl.len().alias("righe"),
    pl.col("mmsi").n_unique().alias("navi"),
    pl.col("mmsi").min().alias("mmsi_min"),
    pl.col("mmsi").max().alias("mmsi_max"),
    pl.col("sog").min().alias("sog_min"),
    pl.col("sog").max().alias("sog_max")
]).collect().row(0)

# Calcoliamo i nulli totali e i messaggi minimi
nulli_totali = sum(lazy_df.null_count().collect().row(0))
min_msg = lazy_df.group_by("mmsi").len().select(pl.col("len").min()).collect().item()

# Output minimalista
print(f"REPORT | Righe: {stats[0]} | Navi uniche: {stats[1]} | Nulli: {nulli_totali}")
print(f"MMSI: {stats[2]} -> {stats[3]} (Regola: 201M - 775M)")
print(f"SOG : {stats[4]} -> {stats[5]} nodi (Regola: 1.0 - 102.3)")
print(f"Messaggi minimi per nave: {min_msg} (Regola: >= 30)")

REPORT | Righe: 68145342 | Navi uniche: 14675 | Nulli: 0
MMSI: 203245890 -> 775998205 (Regola: 201M - 775M)
SOG : 1.0 -> 99.9 nodi (Regola: 1.0 - 102.3)
Messaggi minimi per nave: 30 (Regola: >= 30)
